In [7]:
%pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 3.9 MB/s  0:00:02m 4.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 4.5 MB/s  0:00:04 eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn]0m 5/6 [scikit-learn]]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor


In [3]:
df_clean = pd.read_parquet("../data/processed/electricity_clean.parquet")
temp = pd.read_parquet("../data/processed/air_temperature.parquet")
train_buildings = pd.read_csv("../configs/train_buildings.csv")["building"].tolist()
print(df_clean.shape, temp.shape, len(train_buildings))

(17544, 1578) (17544, 19) 1083


In [4]:
b = "Robin_lodging_Dorthy"
site = b.split("_")[0]
s = df_clean[b]

feat = pd.DataFrame({
    "load": s,
    "hour": s.index.hour,
    "weekday": s.index.weekday,
    "lag24": s.shift(24),
    "lag168": s.shift(168),
    "temp": temp[site],
})
feat.loc["2016-01-08":].head(10)

,load,hour,weekday,lag24,lag168,temp
2016-01-08 00:00:00,33.4,0,4,32.6,14.3,3.3
2016-01-08 01:00:00,28.0,1,4,25.4,27.3,3.2
2016-01-08 02:00:00,23.9,2,4,20.8,25.7,2.6
2016-01-08 03:00:00,20.4,3,4,19.2,22.4,2.7
2016-01-08 04:00:00,19.8,4,4,18.4,20.1,1.9
2016-01-08 05:00:00,19.6,5,4,18.1,18.6,2.4
2016-01-08 06:00:00,19.2,6,4,17.7,17.4,1.6
2016-01-08 07:00:00,20.1,7,4,19.5,19.5,4.2
2016-01-08 08:00:00,24.6,8,4,22.9,22.1,4.4
2016-01-08 09:00:00,32.3,9,4,30.7,26.1,5.8


In [ ]:
features = ["hour", "weekday", "lag24", "lag168", "temp"]

train = feat.loc["2016"].dropna(subset=["load"])
test = feat.loc["2017"].dropna(subset=["load"])

model = HistGradientBoostingRegressor(random_state=0)
model.fit(train[features], train["load"])

pred = model.predict(test[features])
mae_model = (test["load"] - pred).abs().mean()
print("model MAE:", round(mae_model, 2), " | baselines: last week 3.42, yesterday 3.00")